# Zadanie domowe -- interpolacja dwusześcienna

Interpolacja dwusześcienna, to podobnie jak w przypadku interpolacji dwuliniowej, rozszerzenie idei interpolacji jednowymiarowej na dwuwymiarową siatkę.
W trakcie jej obliczania wykorzystywane jest 16 pikseli z otoczenia (dla dwuliniowej 4).
Skutkuje to zwykle lepszymi wynikami - obraz wyjściowy jest bardziej gładki i z mniejszą liczbą artefaktów.
Ceną jest znaczny wzrost złożoności obliczeniowej (zostało to zaobserwowane podczas ćwiczenia).

Interpolacja dana jest wzorem:
\begin{equation}
I(i,j) = \sum_{i=0}^{3} \sum_{j=0}^{3} a_{ij} x^i y^j
\end{equation}

Zadanie sprowadza się zatem do wyznaczenia 16 współczynników $a_{ij}$.
W tym celu wykorzystuje się, oprócz wartość w~puntach $A$ (0,0), $B$ (1 0), $C$ (1,1), $D$ (0,1) (por. rysunek dotyczący interpolacji dwuliniowej), także pochodne cząstkowe $A_x$, $A_y$, $A_{xy}$.
Pozwala to rozwiązać układ 16-tu równań.

Jeśli zgrupujemy parametry $a_{ij}$:
\begin{equation}
a = [ a_{00}~a_{10}~a_{20}~a_{30}~a_{01}~a_{11}~a_{21}~a_{31}~a_{02}~a_{12}~a_{22}~a_{32}~a_{03}~a_{13}~a_{23}~a_{33}]
\end{equation}

i przyjmiemy:
\begin{equation}
x = [A~B~D~C~A_x~B_x~D_x~C_x~A_y~B_y~D_y~C_y~A_{xy}~B_{xy}~D_{xy}~C_{xy}]^T
\end{equation}

To zagadnienie można opisać w postaci równania liniowego:
\begin{equation}
Aa = x
\end{equation}
gdzie macierz $A^{-1}$ dana jest wzorem:

\begin{equation}
A^{-1} =
\begin{bmatrix}
1& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0& 0 \\
0&  0&  0&  0&  1&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0 \\
-3&  3&  0&  0& -2& -1&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0 \\
2& -2&  0&  0&  1&  1&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0 \\
0&  0&  0&  0&  0&  0&  0&  0&  1&  0&  0&  0&  0&  0&  0&  0 \\
0&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0&  0&  1&  0&  0&  0 \\
0&  0&  0&  0&  0&  0&  0&  0& -3&  3&  0&  0& -2& -1&  0&  0 \\
0&  0&  0&  0&  0&  0&  0&  0&  2& -2&  0&  0&  1&  1&  0&  0 \\
-3&  0&  3&  0&  0&  0&  0&  0& -2&  0& -1&  0&  0&  0&  0&  0 \\
0&  0&  0&  0& -3&  0&  3&  0&  0&  0&  0&  0& -2&  0& -1&  0 \\
9& -9& -9&  9&  6&  3& -6& -3&  6& -6&  3& -3&  4&  2&  2&  1 \\
-6&  6&  6& -6& -3& -3&  3&  3& -4&  4& -2&  2& -2& -2& -1& -1 \\
2&  0& -2&  0&  0&  0&  0&  0&  1&  0&  1&  0&  0&  0&  0&  0 \\
0&  0&  0&  0&  2&  0& -2&  0&  0&  0&  0&  0&  1&  0&  1&  0 \\
-6&  6&  6& -6& -4& -2&  4&  2& -3&  3& -3&  3& -2& -1& -2& -1 \\
4& -4& -4&  4&  2&  2& -2& -2&  2& -2&  2& -2&  1&  1&  1&  1 \\
\end{bmatrix}
\end{equation}

Potrzebne w rozważaniach pochodne cząstkowe obliczane są wg. następującego przybliżenia (przykład dla punktu A):
\begin{equation}
A_x = \frac{I(i+1,j) - I(i-1,j)}{2}
\end{equation}

\begin{equation}
A_y = \frac{I(i,j+1) - I(i,j-1)}{2}
\end{equation}

\begin{equation}
A_{xy} = \frac{I(i+1,j+1) - I(i-1,j) - I(i,j-1) + I(i,j)}{4}
\end{equation}

## Zadanie

Wykorzystując podane informacje zaimplementuj interpolację dwusześcienną.
Uwagi:
- macierz $A^{-1}$ dostępna jest w pliku *ainvert.py*
- trzeba się zastanowić nad potencjalnym wykraczaniem poza zakres obrazka (jak zwykle).

Ponadto dokonaj porównania liczby operacji arytmetycznych i dostępów do pamięci koniecznych przy realizacji obu metod interpolacji: dwuliniowej i dwusześciennej.

In [ ]:
import os
import urllib.request
import cv2
import numpy as np
from matplotlib import pyplot as plt
import time

def download_file(url, filename):
    if not os.path.exists(filename):
        urllib.request.urlretrieve(url, filename)

download_file(
    "https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/parrot.bmp",
    "parrot.bmp"
)
download_file(
    "https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/ainvert.py",
    "ainvert.py"
)

import ainvert


def clamp(val, lo, hi):
    return max(lo, min(hi, val))


def get_pixel(img, row, col):
    h, w = img.shape[:2]
    row = clamp(row, 0, h - 1)
    col = clamp(col, 0, w - 1)
    return float(img[row, col])


def compute_derivatives(img, row, col):
    def p(r, c):
        return get_pixel(img, r, c)

    fx  = (p(row, col + 1) - p(row, col - 1)) / 2.0
    fy  = (p(row + 1, col) - p(row - 1, col)) / 2.0
    fxy = (p(row + 1, col + 1) - p(row - 1, col + 1)
           - p(row + 1, col - 1) + p(row - 1, col - 1)) / 4.0
    return fx, fy, fxy


def bicubic_patch_coefficients(img, row0, col0):
    A = get_pixel(img, row0,     col0    )
    B = get_pixel(img, row0,     col0 + 1)
    C = get_pixel(img, row0 + 1, col0 + 1)
    D = get_pixel(img, row0 + 1, col0    )

    Ax, Ay, Axy = compute_derivatives(img, row0,     col0    )
    Bx, By, Bxy = compute_derivatives(img, row0,     col0 + 1)
    Cx, Cy, Cxy = compute_derivatives(img, row0 + 1, col0 + 1)
    Dx, Dy, Dxy = compute_derivatives(img, row0 + 1, col0    )

    x_vec = np.array([
        A,   B,   D,   C,
        Ax,  Bx,  Dx,  Cx,
        Ay,  By,  Dy,  Cy,
        Axy, Bxy, Dxy, Cxy
    ])

    return ainvert.A_invert @ x_vec


def eval_bicubic(a, dx, dy):
    result = 0.0
    for i in range(4):
        for j in range(4):
            result += a[i + 4 * j] * (dx ** i) * (dy ** j)
    return result


def bicubic_image(img, scale_x, scale_y):
    h, w = img.shape[:2]
    new_h = int(h * scale_y)
    new_w = int(w * scale_x)
    result = np.zeros((new_h, new_w), dtype=np.float64)

    for i in range(new_h):
        for j in range(new_w):
            x = j / scale_x
            y = i / scale_y

            col0 = clamp(int(np.floor(x)), 0, w - 2)
            row0 = clamp(int(np.floor(y)), 0, h - 2)
            dx = x - col0
            dy = y - row0

            a = bicubic_patch_coefficients(img, row0, col0)
            result[i, j] = eval_bicubic(a, dx, dy)

    return np.clip(result, 0, 255).astype(np.uint8)


parrot_org = cv2.imread('parrot.bmp', cv2.IMREAD_GRAYSCALE)

t0 = time.perf_counter()
parrot_bilinear = cv2.resize(parrot_org,
                             (parrot_org.shape[1] * 2, parrot_org.shape[0] * 2),
                             interpolation=cv2.INTER_LINEAR)
t_bilinear = time.perf_counter() - t0

t0 = time.perf_counter()
parrot_moja = bicubic_image(parrot_org, 2, 2)
t_bicubic_own = time.perf_counter() - t0

t0 = time.perf_counter()
parrot_cv2 = cv2.resize(parrot_org,
                        (parrot_org.shape[1] * 2, parrot_org.shape[0] * 2),
                        interpolation=cv2.INTER_CUBIC)
t_bicubic_cv2 = time.perf_counter() - t0

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(parrot_org,  cmap='gray')
axes[0].set_title('Oryginał')
axes[0].axis('off')

axes[1].imshow(parrot_moja, cmap='gray')
axes[1].set_title('Moja implementacja (dwusześcienna)')
axes[1].axis('off')

axes[2].imshow(parrot_cv2,  cmap='gray')
axes[2].set_title('OpenCV INTER_CUBIC')
axes[2].axis('off')

plt.tight_layout()
plt.show()

diff = np.abs(parrot_moja.astype(np.int32) - parrot_cv2.astype(np.int32)).astype(np.uint8)

plt.figure(figsize=(7, 6))
plt.imshow(diff, cmap='hot')
plt.colorbar(label='|moja - cv2|')
plt.title('Różnica: moja vs cv2')
plt.axis('off')
plt.show()

print(f"MAE  (mean absolute error): {diff.mean():.4f}")
print(f"Max  błąd:                  {diff.max()}")

print("\n" + "=" * 60)
print("  CZASY WYKONANIA")
print("=" * 60)
print(f"  Dwuliniowa  (cv2 INTER_LINEAR):  {t_bilinear*1000:.1f} ms")
print(f"  Dwusześć.   (cv2 INTER_CUBIC):   {t_bicubic_cv2*1000:.1f} ms")
print(f"  Dwusześć.   (własna, Python):    {t_bicubic_own:.2f} s")
print(f"  Spowolnienie własna vs cv2:      ×{t_bicubic_own/t_bicubic_cv2:.0f}")
print("=" * 60)


